# Figure C.2 — three pictures of the same windows

**In the paper:** Fig. C.2; App. C.2 (Eqs. C.1–C.3).

The two images one can make of a set of windows without a source model, the inverse Radon transform and
faststack (Harrison 2011), beside the target, the scene convolved with the PSF: what an exact linear
inversion of blurred projections could at best return.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the scene and the scan pattern, and the weights δψ_j = π/N | `ins.project_al`, `radon.direction_weights` | App. C.2 |
| 2 | the windows | `sim.simulate_window`; in one call `sim.simulate_campaign` | — |
| 3 | the target, checked against the windows' own model | `ins.render_curve` | App. C.2, Eq. C.1 |
| 4 | the inversion | `radon.ramp_filter`; in one call `radon.filtered_backprojection` | Eq. C.3 |
| 5 | faststack written out for one pixel | `radon.faststack` | Harrison 2011, Eqs. 1–4 |
| 6 | what the pictures show, in numbers | — | App. C.2 |
| 7 | the figure | — | Fig. C.2 |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**The scene.** A triple: A at the origin, B at 150 mas (PA 60°, 𝒮 = 0.40), C at 300 mas (PA 250°,
𝒮 = 0.25), scanned at 36 directions evenly spaced over the half-turn, the complete set an inversion
needs; every window at zero point 0; toy instrument, S/N 100.

**What it reproduces** (the caption of Fig. C.2). Each picture over its value at A: the target, the
inversion and faststack give 0.40, 0.43, 0.47 at B and 0.25, 0.27, 0.24 at C; their largest excursions
farther than 1.5 FWHM from the sources are 0.002, 0.032 and 0.018. A few seconds.

In [ ]:
# the imports: numpy, matplotlib, scipy, and the scanlib modules; every scanlib call below carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import radon                  # the inverse Radon transform and faststack

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in auxil/sim_margins_lambda_w)
FWHM_MAS, PITCH_MAS, N_SAMPLES = 100.0, 59.0, 18
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)

SAVE_PDF = False                           # True: write pdf/figC2_three_pictures.pdf

In [ ]:
# the triple scene and its noise, the 36 scan directions, the image grid and the range of the cuts
# SOURCES: a table from a source's name to three numbers: (separation [mas], position angle [deg, north through
# east], flux ratio to the primary); SOURCES["B"] is (150.0, 60.0, 0.40)
SOURCES = {"A": (0.0, 0.0, 1.0),
           "B": (150.0, 60.0, 0.40),
           "C": (300.0, 250.0, 0.25)}
NAMES = tuple(SOURCES)                     # the names in the table's order: ("A", "B", "C")
# each source's sky offset (dalpha*, ddelta) = (rho sin PA, rho cos PA) [mas], and its flux ratio, in the same order
offsets_list = []
flux_ratio_list = []
for rho_mas, pa_deg, flux_ratio in SOURCES.values():
    offsets_list.append([rho_mas * np.sin(np.deg2rad(pa_deg)), rho_mas * np.cos(np.deg2rad(pa_deg))])
    flux_ratio_list.append(flux_ratio)
OFFSETS = np.array(offsets_list)           # one row per source, two columns: dalpha*, ddelta [mas]
FLUX_RATIOS = np.array(flux_ratio_list)    # one entry per source
SCENE = sim.Scene(OFFSETS, FLUX_RATIOS * sim.PRIMARY_FLUX)
FLUXES = SCENE.fluxes                      # the three fluxes [toy flux units]
SEED = 20260918                            # the noise seed
SIGMA_N = sim.PRIMARY_FLUX / 100.0             # the toy regime, S/N 100: the noise sigma per sample
N_ANGLES = 36                              # directions, evenly spaced over [0, pi)
PSI = np.arange(N_ANGLES) * np.pi / N_ANGLES   # the scan angles [rad]
GRID_HALF_MAS = 400.0                      # the image square's half-width [mas]
GRID_STEP_MAS = 4.0                        # the image's pixel [mas]
FASTSTACK_ALPHA = 0.05                     # the fraction of lowest samples that sets a pixel's level
CUT_RANGE_MAS = (-150.0, 420.0)            # the cuts' range of distance from the primary

## 1. The scene and the scan pattern (App. C.2)

Along each scan the three sources project to η = Δα\* sin ψ + Δδ cos ψ (Eq. 9); along some directions two
or three of them fall within one PSF. 36 directions evenly spaced over [0°, 180°): each weight is π/36.

In [ ]:
# Eq. 9 for every source at every direction: one row per direction, one column per source
eta_rows = []
for psi_j in PSI:
    # this direction's row: the along-scan position of each source [mas]
    eta_row = []
    for source_x, source_y in OFFSETS:
        eta_row.append(ins.project_al(source_x, source_y, psi_j))          # ins.project_al: Eq. 9
    eta_rows.append(eta_row)
ETA_TRUE = np.array(eta_rows)
# sim.Scene.along_scan: the same projection, direction by direction
assert np.allclose(ETA_TRUE, np.array([SCENE.along_scan(psi_j) for psi_j in PSI]))
# radon.direction_weights: delta psi_j, each direction's share of pi
WEIGHTS = radon.direction_weights(PSI)
assert np.allclose(WEIGHTS, np.pi / PSI.size)
for name, (source_x, source_y), flux_ratio in zip(NAMES, OFFSETS, FLUX_RATIOS):
    print(f"{name}: (dalpha*, ddelta) = ({source_x:7.1f}, {source_y:7.1f}) mas, flux ratio {flux_ratio:.2f}")
# pairs closer than one FWHM along the scan, counted over the directions; ETA_TRUE[:, i] keeps every direction
# (row) and the one column of source i
# PAIRS: a table from a pair's name to the numbers of its two sources (columns of ETA_TRUE); PAIRS["A-B"] is (0, 1)
PAIRS = {"A-B": (0, 1), "A-C": (0, 2), "B-C": (1, 2)}
# close: a table from a pair's name to the number of directions along which the pair lies within one FWHM
close = {}
for pair, (i, j) in PAIRS.items():
    separation_along_scan = np.abs(ETA_TRUE[:, i] - ETA_TRUE[:, j])
    close[pair] = int(np.sum(separation_along_scan < FWHM_MAS))
# the table as text: "A-B 5, A-C 3, ..." (name, count)
close_texts = []
for pair, count in close.items():
    close_texts.append(f"{pair} {count}")
print(f"{PSI.size} directions at {180 / PSI.size:.0f} deg pitch, each weighted pi/{PSI.size}; "
      "directions along which a pair lies within one FWHM: " + ", ".join(close_texts))

## 2. The windows

S/N 100 from the notebook's seed, every window at zero point 0; `sim.simulate_campaign` renders the same.

In [ ]:
# the 36 noisy windows, by hand and by sim.simulate_campaign from the same seed, every zero point 0
# rng_noise: the generator of the windows' noise, seeded by SEED alone; the windows draw from it in the order of
# the directions
rng_noise = np.random.default_rng(SEED)
WINDOWS = []
for psi_j in PSI:
    # sim.simulate_window: model + one noise draw
    samples = sim.simulate_window(SCENE.along_scan(psi_j), FLUXES, inst, rng_noise, SIGMA_N)
    WINDOWS.append(samples)
WINDOWS = np.array(WINDOWS)                                           # (J, N)
# sim.simulate_campaign: every window of the law in one call: (windows, their noise sigma, the true positions);
# the true positions are not used here. A new generator from the same seed, so that it draws the same noise
rng_noise_one_call = np.random.default_rng(SEED)
package_windows, package_sigma, _eta_true = sim.simulate_campaign(SCENE, PSI, rng_noise_one_call, inst, SIGMA_N)
assert np.array_equal(np.array(package_windows), WINDOWS) and package_sigma == SIGMA_N
print(f"{WINDOWS.shape[0]} windows x {WINDOWS.shape[1]} samples, noise sigma {SIGMA_N:.2f} per sample; "
      "sim.simulate_campaign renders the same windows")

## 3. The target (App. C.2)

The scene convolved with the PSF, in flux per mas². Integrated across the scan it must give the windows'
model profile (Eq. C.1, flux per sample over the pitch): checked at ψ = 90° and 0°.

In [ ]:
# The target on the image grid: the scene convolved with the Gaussian PSF [flux per mas^2].
AXIS = np.arange(-GRID_HALF_MAS, GRID_HALF_MAS + 1e-9, GRID_STEP_MAS)   # pixel centres [mas], both axes
X, Y = np.meshgrid(AXIS, AXIS)                                           # X = dalpha* (columns), Y = ddelta (rows)
I0 = int(np.argmin(np.abs(AXIS)))                                        # the pixel at the origin
SIGMA_PSF = FWHM_MAS / (2.0 * np.sqrt(2.0 * np.log(2.0)))            # the Gaussian PSF's sigma [mas]
PITCH = float(np.diff(inst.sample_grid_mas)[0])                           # the sample pitch [mas]
# one Gaussian of unit integral per source, times the source's flux
TARGET = np.zeros_like(X)
for (source_x, source_y), flux in zip(OFFSETS, FLUXES):
    distance_squared = (X - source_x) ** 2 + (Y - source_y) ** 2
    TARGET += flux * np.exp(-0.5 * distance_squared / SIGMA_PSF ** 2) / (2 * np.pi * SIGMA_PSF ** 2)

In [ ]:
# the projection theorem: the target integrated across the scan is the model profile along it;
# integrated on a wider grid (+-700 mas), so that no source's wings are cut by the image's edge
wide = np.arange(-700.0, 700.0 + 1e-9, GRID_STEP_MAS)                    # the wider grid's pixel centres [mas]
XW, YW = np.meshgrid(wide, wide)                                         # XW = dalpha* (columns), YW = ddelta (rows)
# the target on the wider grid: the same sum of Gaussians as TARGET
target_wide = np.zeros_like(XW)
for (source_x, source_y), flux in zip(OFFSETS, FLUXES):
    distance_squared = (XW - source_x) ** 2 + (YW - source_y) ** 2
    target_wide += flux * np.exp(-0.5 * distance_squared / SIGMA_PSF ** 2) / (2 * np.pi * SIGMA_PSF ** 2)
# sum(axis=0) sums over the rows (ddelta) and leaves the profile along dalpha*, the scan at psi = 90 deg;
# sum(axis=1) sums over the columns (dalpha*) and leaves the profile along ddelta, the scan at psi = 0
# the pixel size is the integral's step, so each projection is in flux per mas
projection_90 = target_wide.sum(axis=0) * GRID_STEP_MAS
projection_0 = target_wide.sum(axis=1) * GRID_STEP_MAS
for psi_deg, projection in ((90.0, projection_90), (0.0, projection_0)):
    # ins.render_curve: the windows' noise-free model profile at any eta [flux per sample]; over the pitch, per mas
    model = ins.render_curve(wide, SCENE.along_scan(np.deg2rad(psi_deg)), FLUXES, inst) / PITCH
    # the toy LSF is the Gaussian by the midpoint rule: 1e-5 off
    assert np.allclose(projection, model, rtol=1e-4, atol=1e-6 * model.max())
print(f"target on a {AXIS.size} x {AXIS.size} grid of {GRID_STEP_MAS:.0f} mas pixels; "
      "its projections at 0 and 90 deg equal the windows' model profile")

## 4. The inversion (Eq. C.3)

$$I(\Delta\alpha^*, \Delta\delta) \approx \sum_j q\!\left(\Delta\alpha^*\sin\psi_j + \Delta\delta\cos\psi_j \mid \psi_j\right)\delta\psi_j,
\qquad \hat q(\nu\mid\psi) \equiv |\nu|\,\hat s(\nu\mid\psi), \qquad ({\rm C.3})$$

as in `figC1_inverse_radon.ipynb` §4. The result is the PSF-blurred scene, band-limited by the sampling,
with the grain of 36 directions.

In [ ]:
# The inverse Radon transform by hand, checked against radon.filtered_backprojection.
FBP = np.zeros_like(X)
for psi_j, weight_j, samples in zip(PSI, WEIGHTS, WINDOWS):
    # radon.ramp_filter: the ramp-filtered projection: the positions of its fine grid eta_fine [mas] and the
    # filtered values q (the q of Eq. C.3)
    eta_fine, q = radon.ramp_filter(samples, inst.sample_grid_mas)
    pixel_eta = X * np.sin(psi_j) + Y * np.cos(psi_j)                    # every pixel's coordinate along this scan
    # each pixel collects the filtered value at its coordinate, weighted; outside the window it collects zero
    FBP += weight_j * np.interp(pixel_eta, eta_fine, q, left=0.0, right=0.0)
# radon.filtered_backprojection: the package's inverse Radon transform, the same sum in one call
package_image, package_axis = radon.filtered_backprojection(PSI, WINDOWS, inst, half=GRID_HALF_MAS, step=GRID_STEP_MAS)
assert np.allclose(package_axis, AXIS) and np.allclose(package_image, FBP, rtol=0, atol=1e-12 * np.abs(FBP).max())
print(f"inverse Radon transform by hand == radon.filtered_backprojection (to {np.abs(package_image - FBP).max() / np.abs(FBP).max():.1e} of the peak)")

## 5. Faststack (Harrison 2011, Eqs. 1–4)

Each pixel collects, from every window, the one sample whose strip on the sky contains it, and sorts
the $N$ values $\theta_1 \le \dots \le \theta_N$. The lowest fraction α = 5 % sets a first level $r_0$
(their mean); every higher sample is weighted down,

$$w_k = 1\ \ (k \le \alpha N), \qquad w_k = e^{\,1 - (\theta_k / r_0)^2}\ \ (k > \alpha N),$$

and the pixel is the weighted mean. The image is in flux per sample, not a calibrated flux per mas²; at
one sampling phase it is blocky.

In [ ]:
# Faststack written out for one pixel (Harrison 2011, Eqs. 1-3).
GRID = inst.sample_grid_mas                # the positions of a window's samples along the scan [mas]


def faststack_pixel(x0, y0, alpha=FASTSTACK_ALPHA):
    """The value of one faststack pixel, Harrison (2011) Eqs. 1-3 written out.

    Inputs
        x0, y0   the pixel's sky offset (dalpha*, ddelta) [mas]
        alpha    the fraction of the lowest samples that sets the pixel's first level
    Returns one number: the weighted mean of the samples the pixel collects [flux per sample]."""
    # step 1, collect: from every window, the one sample whose strip on the sky contains the pixel
    theta = []                             # Harrison's theta: the collected samples
    for psi_j, window in zip(PSI, WINDOWS):
        # the pixel's place along this window's scan, and the sample whose strip contains it: the distance from
        # the first sample (GRID[0]) in units of the pitch, rounded to the nearest whole number
        sample_index = int(np.floor((x0 * np.sin(psi_j) + y0 * np.cos(psi_j) - GRID[0]) / PITCH + 0.5))
        # the pixel lies outside this window when the index is not one of its samples
        if 0 <= sample_index < GRID.size:
            theta.append(window[sample_index])
    # step 2, sort: theta_1 <= ... <= theta_N
    theta = np.sort(theta)
    # step 3, the first level r0: the mean of the n_lowest lowest values (at least one)
    n_lowest = max(1, int(alpha * theta.size))
    r0 = theta[:n_lowest].mean()                          # Eq. 3: the level set by the lowest values
    # step 4, the weights w_k: 1 for the n_lowest lowest values, smaller for the higher ones ([n_lowest:])
    w = np.ones(theta.size)
    w[n_lowest:] = np.exp(1.0 - (theta[n_lowest:] / r0) ** 2)   # Eq. 2: down-weight the samples that caught a bright source
    # step 5, the pixel: the weighted mean
    return np.sum(w * theta) / np.sum(w)                  # Eq. 1

In [ ]:
# the full image on the same grid; the pixel by hand is checked against it at the three sources and one empty
# pixel
# radon.faststack: the package's image, with its variance (not used here), the samples per pixel and the axis
FS, fs_variance, fs_count, axis_fs = radon.faststack(PSI, WINDOWS, inst, half=GRID_HALF_MAS, step=GRID_STEP_MAS, alpha=FASTSTACK_ALPHA)
assert np.allclose(axis_fs, AXIS)
# the three sources' offsets (*OFFSETS: its three rows) and one empty place, (-300, 300) mas
for x0, y0 in (*OFFSETS, (-300.0, 300.0)):
    # row: the pixel row nearest to ddelta = y0; column: the pixel column nearest to dalpha* = x0
    row = np.argmin(np.abs(AXIS - y0))
    column = np.argmin(np.abs(AXIS - x0))
    assert np.isclose(faststack_pixel(AXIS[column], AXIS[row]), FS[row, column], rtol=1e-12)
print("faststack by hand == radon.faststack at the three sources and one empty pixel; "
      f"samples per pixel {int(np.nanmin(fs_count))}-{int(np.nanmax(fs_count))}")

## 6. What the pictures show (App. C.2)

Each picture over its value at A: its value at B and at C; the dip between A and B; the deepest trough;
the grain, the largest excursion farther than 1.5 FWHM from every source.

In [ ]:
# The three pictures, each over its value at A; the mask of the pixels far from every source; and the cut of a
# picture along a line through A.
# PICTURES: a table from a picture's name to its image over the image's value at A (the pixel [I0, I0], the origin)
PICTURES = {"target": TARGET / TARGET[I0, I0], "inverse Radon": FBP / FBP[I0, I0], "faststack": FS / FS[I0, I0]}
# FAR: the mask of the pixels farther than 1.5 FWHM from every source: one mask per source, then the pixels that
# are in all three (np.all along axis 0, the axis of the sources)
far_from_each = []
for source_x, source_y in OFFSETS:
    far_from_each.append(np.hypot(X - source_x, Y - source_y) > 1.5 * FWHM_MAS)
FAR = np.all(far_from_each, axis=0)        # far from every source


def cut(img, target_idx, r):
    """The image along the line from A through one source.

    Inputs
        img          the image, on the grid AXIS x AXIS (rows along ddelta, columns along dalpha*)
        target_idx   the source the line passes through: its number in OFFSETS (1 is B, 2 is C)
        r            the distances from A along the line [mas]; negative: on the side away from the source
    Returns one value of the image per distance, interpolated linearly between the four nearest pixels."""
    # the unit vector from A (the origin) to the source
    direction = OFFSETS[target_idx] / np.hypot(*OFFSETS[target_idx])
    # the points of the line in pixel units: the offset from the first pixel centre (AXIS[0]) over the pixel size
    columns = (r * direction[0] - AXIS[0]) / GRID_STEP_MAS               # from dalpha*
    rows = (r * direction[1] - AXIS[0]) / GRID_STEP_MAS                  # from ddelta
    # map_coordinates: the image at (row, column) places between the pixels; order=1 is linear interpolation
    return map_coordinates(img, [rows, columns], order=1, mode="nearest")

In [ ]:
# truth: each picture at the true places of B and C, the dip between A and B, the deepest trough, and the grain
R_CUT = np.linspace(*CUT_RANGE_MAS, 571)   # the cuts' distances from A [mas]: 571 points from -150 to 420, 1 mas apart
# NUMBERS: a table from a picture's name to that picture's own table of five numbers: "B", "C", "dip", "trough"
# and "grain"
NUMBERS = {}
for name, img in PICTURES.items():
    # at: a table from a source's name to the picture's value at the source's nearest pixel (row from ddelta,
    # column from dalpha*)
    at = {}
    for source_name, (source_x, source_y) in zip(NAMES, OFFSETS):
        row = np.argmin(np.abs(AXIS - source_y))
        column = np.argmin(np.abs(AXIS - source_x))
        at[source_name] = img[row, column]
    rho_b = np.hypot(*OFFSETS[1])          # B's separation from A [mas]; OFFSETS[1] is B's row
    # the cut towards B (source 1); the mask keeps the part between A (r = 0) and B (r = rho_b) only
    between = cut(img, 1, R_CUT)[(R_CUT > 0) & (R_CUT < rho_b)]
    NUMBERS[name] = dict(B=at["B"], C=at["C"], dip=between.min() / at["B"], trough=img.min(), grain=np.abs(img[FAR]).max())
    print(f"{name:14s}: at B {at['B']:.3f}, at C {at['C']:.3f}; dip between A and B {NUMBERS[name]['dip']:.2f} of its value at B; "
          f"deepest trough {img.min():+.3f}; grain beyond 1.5 FWHM {NUMBERS[name]['grain']:.3f}")
# faststack has no ramp troughs: its most negative value is its noise floor, within its grain
assert -NUMBERS["faststack"]["trough"] <= NUMBERS["faststack"]["grain"] < -NUMBERS["inverse Radon"]["trough"]
assert NUMBERS["faststack"]["grain"] < NUMBERS["inverse Radon"]["grain"]

## 7. The figure (Fig. C.2)

Top: the three pictures on one grid and one grey scale. Bottom: the cuts from A through B and through C,
one line style per picture. Plotting only.

In [ ]:
# Top row: the three pictures on one grid and one grey scale; bottom row: the two cuts, from A
# through B and through C, the three pictures as three line styles.
# --- FAST cell: plotting only. ---
plt.rcParams.update({"font.size": 10.0, "axes.linewidth": 0.5,
                     "xtick.labelsize": 9.0, "ytick.labelsize": 9.0})
PRIMARY_MK = dict(marker="+", s=80, color="white", lw=1.2, zorder=6)
TRUTH_MK = dict(marker="*", s=70, facecolor="white", edgecolor="k", lw=0.6, zorder=6)
# one line style per picture
STYLE = {"target": dict(ls="--", color="k", lw=0.7),
         "inverse Radon": dict(ls="-", color="0.55", lw=0.8),
         "faststack": dict(ls="-", color="k", lw=0.8)}
EXTENT = (AXIS[0], AXIS[-1], AXIS[0], AXIS[-1])
VIEW_MAS = 370.0                                          # the pictures are shown to this radius (the grid runs to 400)

fig = plt.figure(figsize=(7.0, 3.85))
gs = fig.add_gridspec(2, 6, height_ratios=[2.25, 1.0], hspace=0.38, wspace=0.5,
                      left=0.075, right=0.995, top=0.985, bottom=0.125)
# the three picture panels: each takes two of the grid's six columns in the top row
top = [fig.add_subplot(gs[0, 2 * i:2 * i + 2]) for i in range(3)]
for ax, (name, img), lab in zip(top, PICTURES.items(), "abc"):
    ax.imshow(img, origin="lower", extent=EXTENT, cmap="gray_r", vmin=0.0, vmax=1.0,
              interpolation="nearest", rasterized=True)
    ax.set_xlim(VIEW_MAS, -VIEW_MAS)                                     # East to the left
    ax.set_ylim(-VIEW_MAS, VIEW_MAS)
    ax.scatter(0, 0, **PRIMARY_MK)
    # [1:] omits the first row (A, marked above): B and C only; column 0 is dalpha*, column 1 ddelta
    ax.scatter(OFFSETS[1:, 0], OFFSETS[1:, 1], **TRUTH_MK)
    for source_name, (source_x, source_y) in zip(NAMES, OFFSETS):     # the sources' names
        ax.annotate(source_name, (source_x, source_y), xytext=(-9, 7), textcoords="offset points", fontsize=9,
                    fontweight="bold")
    ax.text(0.04, 0.96, rf"$\bf{{({lab})}}$ {name}", transform=ax.transAxes, va="top", ha="left", fontsize=9,   # bold letter, house style
            bbox=dict(boxstyle="round", fc="white", ec="none", alpha=0.85))
    ax.set_xticks([300, 0, -300])
    ax.set_yticks([-300, 0, 300])
    ax.tick_params(direction="in", color="0.5")
    ax.set_xlabel(r"$\Delta\alpha^{*}$ [mas]  (East $\leftarrow$)")
top[0].set_ylabel(r"$\Delta\delta$ [mas]")
for ax in top[1:]:
    ax.set_yticklabels([])

# the two cut panels: each takes three of the six columns in the bottom row
bottom = [fig.add_subplot(gs[1, 0:3]), fig.add_subplot(gs[1, 3:6])]
# idx: the source the cut passes through, its number in OFFSETS (1 is B, 2 is C)
for ax, idx, lab in zip(bottom, (1, 2), "de"):
    for name, img in PICTURES.items():
        ax.plot(R_CUT, cut(img, idx, R_CUT), label=name, **STYLE[name])
    ax.axhline(0, color="0.8", lw=0.5, zorder=0)
    ax.axvline(np.hypot(*OFFSETS[idx]), color="0.8", lw=0.5, ls=":", zorder=0)
    ax.set_xlim(*CUT_RANGE_MAS)
    ax.set_ylim(-0.12, 1.08)
    ax.set_xlabel(f"distance from A towards {NAMES[idx]} [mas]")
    ax.text(0.03, 0.93, f"({lab})", transform=ax.transAxes, va="top", ha="left", fontsize=9, fontweight="bold")
    ax.tick_params(direction="in")
bottom[0].set_ylabel("value / value at A")
bottom[1].set_yticklabels([])
bottom[0].legend(loc="upper right", fontsize=8, frameon=False, handlelength=2.2)

In [ ]:
# each picture's numbers once more, for the caption; the caption's numbers asserted
# numbers: one picture's table (its entries are listed where NUMBERS is made)
for name, numbers in NUMBERS.items():
    print(f"{name:14s}: B {numbers['B']:.3f}, C {numbers['C']:.3f}, dip {numbers['dip']:.2f}, trough {numbers['trough']:+.3f}, grain {numbers['grain']:.3f}")

# the caption's numbers: one value per picture, in the order target, inverse Radon, faststack
assert [round(NUMBERS[name]["B"], 2) for name in PICTURES] == [0.40, 0.43, 0.47]
assert [round(NUMBERS[name]["C"], 2) for name in PICTURES] == [0.25, 0.27, 0.24]
assert [round(NUMBERS[name]["grain"], 3) for name in PICTURES] == [0.002, 0.032, 0.018]
print("checks passed")

In [ ]:
# write the PDF only when SAVE_PDF is True
if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "figC2_three_pictures.pdf", bbox_inches="tight")
    print("wrote pdf/figC2_three_pictures.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")